# Логистическая регрессия

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from ipywidgets import interact, FloatLogSlider, FloatSlider
from setup import *

## Геометрическая интерпретация

Логистическую регрессию можно рассматривать как способ построения линейной разделяющей поверхности между классами. В случае бинарной классификации мы ищем гиперплоскость (в двумерном пространстве — прямую), которая наилучшим образом отделяет объекты одного класса от другого.

Разделяющая поверхность задается уравнением:

$$w^\top x = 0$$

Знак этого выражения определяет, к какому классу мы относим объект:

* Если $w^\top x > 0$, то $x$ принадлежит классу $+1$ 
* Если $w^\top x < 0$, то $x$ принадлежит классу $-1$

Геометрически вектор весов $w$ является нормалью к разделяющей гиперплоскости. Расстояние от точки $x$ до гиперплоскости равно $\frac{|w^\top x|}{\|w\|_2}$. Чем дальше точка от гиперплоскости, тем увереннее классификатор в своем предсказании.

![alttext](../data/20170513_gradient_descent_logistic_animation.gif)

### Линейные классификаторы

Пусть модель задаётся следующим равенством
$$ a(x, w) = sign \ f(x,w),$$

где $w$ - вектор параметров. $f(x,w)$ - `дискриминантная функция`

Если $f(x, w) > 0$, то алгоритм $a$ относит объект $x$ к классу $+1$, иначе к классу $-1$.

Само уравнение $f(x, w) = 0$ описывает разделяющую гиперплоскость

Понятие **отступа (margin)** тесно связано с геометрической интерпретацией классификации. Для бинарной классификации с метками $y \in \{-1, +1\}$ отступ для объекта $x_i$ определяется как:

$$M_i = y_i f(x_i, w)$$

Физический смысл отступа:
* Если $M_i > 0$ — объект классифицирован **верно** (чем больше = тем надёжнее классификация)
* Если $M_i < 0$ — объект классифицирован **неверно**
* Абсолютное значение $|M_i|$ пропорционально расстоянию от объекта до разделяющей поверхности

Чем больше отступ, тем увереннее классификатор в своем предсказании.

![alt](../data/margin2.webp)

![alt](../data/margin.png)

### А что там в задаче бин. классификации???
В задаче бинарной классификации мы ищем веса $w$, минимизируя эмпирический риск:
$$\mathcal{L}(w) = \frac{1}{N} \sum_{i=1}^\mathcal{l}[y_if(x_i, w) < 0]$$

Но как нам тогда его минимизировать, она ведь разрывна?
Поскольку она разрывна, мы заменяем её на выпуклую верхнюю оценку $\phi(M)$, где $M = y_i (w^\top x_i)$ — **отступ (margin)**:

$$\mathcal{L} = \sum_{i=1}^\mathcal{l}[y_if(x_i, w) < 0] = \sum_{i=1}^\mathcal{l}[M < 0] \leq \sum_{i=1}^\mathcal{l}\phi(y_if(x_i, w))$$


На что влияет выбор $\phi$:
1. **Логистическая функция**: $\ln(1 + \exp(-M))$ — дает Логистическую регрессию.
2. **Hinge Loss**: $\max(0, 1 - M)$ — дает Метод опорных векторов (SVM).
3. **Экспоненциальная**: $\exp(-M)$ — используется в AdaBoost.



<div style='background-color: #ffeb3b; padding: 15px; border-left: 5px solid #ffc107; margin: 10px 0'>
<b>💡 вопрос:</b> Как будем искать оптимум?
</div>


В логистической регрессии мы не можем использовать простую точность (accuracy) в качестве функции потерь, потому что это кусочно-постоянная функция, которая не подходит для градиентного спуска. Вместо этого мы используем функцию потерь, основанную на принципе максимума правдоподобия.

Для бинарной классификации с целевыми метками $y \in \{0, 1\}$ модель предсказывает вероятность принадлежности к классу $1$:

$$\hat{y} = \sigma(w^\top x) = \frac{1}{1 + e^{-w^\top x}}$$

Правдоподобие для одного объекта:

$$p(y | x, w) = \hat{y}^y (1 - \hat{y})^{1-y}$$

Логарифмируя и меняя знак, получаем функцию потерь для одного объекта — **логистическую функцию потерь (log-loss)**:

$$\mathcal{L}(w, x_i, y_i) = -y_i \log \hat{y}_i - (1 - y_i) \log(1 - \hat{y}_i)$$

Итоговая функция потерь на всем датасете — это **средняя перекрестная энтропия (cross-entropy)**:

$$L(w) = -\frac{1}{N}\sum_{i=1}^N \left[ y_i \log \hat{y}_i + (1 - y_i) \log(1 - \hat{y}_i) \right]$$

Эта функция является гладкой и выпуклой, что позволяет эффективно находить оптимальные веса $w$ с помощью градиентных методов.


 В контексте логистической регрессии отступ связан с вероятностью:

$$\hat{y} = \sigma(M_i)$$

Логистическую функцию потерь можно переписать через отступ:

$$\mathcal{L}(M_i) = \log(1 + e^{-M_i})$$
$$\mathcal{L}(M_i) = -y_i \log\sigma(w^\top x_i) - (1-y_i)\log(1-\sigma(w^\top x_i))$$

Для меток $y \in \{0, 1\}$ аналогичное выражение:


### Функция потерь

 В задаче бинарной классификации нашей конечной целью является минимизация количества ошибок, то есть **эмпирического риска**:
$$\mathcal{L}(w) = \frac{1}{N} \sum_{i=1}^{N} [M_i < 0], \quad \text{где } M_i = y_i f(x_i, w)$$
Здесь $M_i$ — **отступ (margin)**. Если $M_i < 0$, объект классифицирован неверно. Однако функция $[M < 0]$ является кусочно-постоянной: её производная везде либо равна нулю, либо не существует. Это делает невозможным применение градиентных методов оптимизации.

### 1. Аппроксимация эмпирического риска
Чтобы сделать задачу решаемой, мы заменяем разрывную функцию потерь на её гладкую выпуклую верхнюю оценку $\phi(M)$:
$$\sum_{i=1}^{N} [M_i < 0] \leq \sum_{i=1}^{N} \phi(M_i)$$
Выбор функции $\phi(M)$ определяет характер алгоритма:
1.  **Логистическая**: $\ln(1 + \exp(-M))$ — приводит к логистической регрессии.
2.  **Hinge Loss**: $\max(0, 1 - M)$ — приводит к методу опорных векторов (SVM).
3.  **Экспоненциальная**: $\exp(-M)$ — используется в алгоритме AdaBoost.

![alt](../data/graphics.png)


### 2. Вероятностная интерпретация и сигмоида
Почему в логистической регрессии используется именно $\ln(1 + e^{-M})$? Это вытекает из естественной вероятностной модели. По теореме Байеса апостериорная вероятность класса $C_1$ при условии признаков $x$ равна:
$$p(C_1 | x) = \frac{p(x | C_1)p(C_1)}{p(x | C_1)p(C_1) + p(x | C_2)p(C_2)} = \frac{1}{1 + \exp(-a)} = \sigma(a)$$
где $a = \ln \frac{p(x | C_1)p(C_1)}{p(x | C_2)p(C_2)}$ — логарифм отношения правдоподобий. Если предположить, что данные внутри классов распределены нормально с общей ковариационной матрицей, то $a$ становится линейной функцией весов: $a = w^\top x$. Таким образом, предсказание модели принимает вид $\hat{y} = \sigma(w^\top x)$.

### 3. От правдоподобия к Log-Loss
Для поиска оптимальных весов мы используем **принцип максимума правдоподобия**. Для меток $y \in \{0, 1\}$ правдоподобие одного объекта записывается как $p(y | x, w) = \hat{y}^y (1 - \hat{y})^{1-y}$. Минимизация отрицательного логарифма правдоподобия дает нам **среднюю перекрестную энтропию (cross-entropy)**:
$$L(w) = -\frac{1}{N}\sum_{i=1}^N \left[ y_i \log \hat{y}_i + (1 - y_i) \log(1 - \hat{y}_i) \right]$$
Используя свойства сигмоиды и перейдя к меткам $y \in \{-1, +1\}$:

Давайте посмотрим [подробнее](https://mlu-explain.github.io/logistic-regression/)

![alttext](../data/logistic-regression.gif)

#### 1. Свойства сигмоиды
Заметим важное свойство симметрии функции $\sigma(z) = \frac{1}{1 + e^{-z}}$:
$$1 - \sigma(z) = 1 - \frac{1}{1 + e^{-z}} = \frac{1 + e^{-z} - 1}{1 + e^{-z}} = \frac{e^{-z}}{1 + e^{-z}} = \frac{1}{e^z + 1} = \sigma(-z)$$
Таким образом, $1 - \sigma(w^\top x) = \sigma(-w^\top x)$.

#### 2. Унификация меток $y \in \{+1, -1\}$
В исходной кросс-энтропии используются метки $y \in \{0, 1\}$. Перейдем к меткам $y \in \{+1, -1\}$. 
Теперь вероятность того, что модель предскажет правильный класс $y$, можно записать одной строкой:
$$P(y \mid x, w) = \begin{cases} \sigma(w^\top x), & \text{если } y = +1 \\ \sigma(-w^\top x), & \text{если } y = -1 \end{cases} = \sigma(y \cdot w^\top x)$$
Величина $M = y \cdot w^\top x$ и есть **отступ (margin)**. Если он положителен, модель права, если отрицателен — ошиблась.

#### 3. Логарифмирование
Теперь возьмем отрицательный логарифм от этого унифицированного правдоподобия:
$$\mathcal{L}(M) = -\ln \sigma(M) = -\ln \left( \frac{1}{1 + e^{-M}} \right)$$
Используя свойство логарифма частного $\ln(\frac{1}{a}) = -\ln(a)$, получаем:
$$\mathcal{L}(M) = - ( - \ln(1 + e^{-M}) ) = \ln(1 + e^{-M})$$

$$\mathcal{L}(M) = \log(1 + e^{-M})$$
Эта функция гладкая и выпуклая, что позволяет эффективно находить глобальный минимум весов $w$ с помощью градиентного спуска.

### 4. Связь с теорией информации
С точки зрения теории информации, минимизация кросс-энтропии эквивалентна минимизации **расстояния Кульбака — Лейблера (KL-дивергенции)** между истинным распределением данных $p_{data}$ и распределением, предсказываемым моделью $q$:
$$KL(p_{data} \| q) = H(p_{data}, q) - H(p_{data})$$
Поскольку энтропия реальных данных $H(p_{data})$ неизменна, подбор весов модели $q(w)$ минимизирует "информационную разницу" между предсказаниями и реальностью. В этом контексте логистическая регрессия находит наиболее правдоподобное распределение, максимально близкое к эмпирическому.


## Многоклассловый случай

Логистическая регрессия естественно обобщается на случай с $K$ классами. Такое обобщение называется **многоклассовой логистической регрессией** или **softmax-регрессией**.

В этом случае для каждого класса $k$ у нас есть свой вектор весов $w_k$. Вероятность принадлежности объекта $x$ к классу $k$ вычисляется с помощью функции **softmax**:

$$p(y = k | x, W) = \frac{\exp(w_k^\top x)}{\sum_{j=1}^K \exp(w_j^\top x)}$$

Функция softmax превращает $K$ действительных чисел (логитов) в $K$ вероятностей, сумма которых равна 1.

Функция потерь для многоклассового случая — это **категориальная кросс-энтропия (categorical cross-entropy)**. Для одного объекта с истинной меткой $y$ (в one-hot кодировке) и предсказанными вероятностями $\hat{y}_k$:

$$\mathcal{L}(W, x_i, y_i) = -\sum_{k=1}^K y_{ik} \log \hat{y}_{ik}$$

На всем датасете:

$$\mathcal{L}(W) = -\frac{1}{N}\sum_{i=1}^N \sum_{k=1}^K y_{ik} \log \frac{\exp(w_k^\top x_i)}{\sum_{j=1}^K \exp(w_j^\top x_i)}$$

Эта функция является гладкой и выпуклой, что позволяет эффективно находить оптимальные веса для всех классов одновременно.

Геометрически в многоклассовом случае мы строим $K$ разделяющих гиперплоскостей (по одной на каждый класс). Область пространства, соответствующая классу $k$, определяется условием $w_k^\top x > w_j^\top x$ для всех $j \neq k$. Эти области разделены кусочно-линейными границами.

![alt](../data/cat.webp)

# Регуляризация

## Теор.мин

В машинном обучении **норма** — это функция, которая измеряет «длину» или «размер» вектора. Если у нас есть вектор весов $w = (w_1, w_2, \dots, w_d)$, то норма $\|w\|$ превращает этот набор чисел в одно неотрицательное число. Чем больше компоненты вектора, тем больше его норма.

### Математическое определение
Формально, нормой в векторном пространстве называется функция $\|w\|$, удовлетворяющая трем аксиомам:
1.  **Неотрицательность**: $\|w\| \geq 0$, причем $\|w\| = 0$ только если $w = \mathbf{0}$.
2.  **Однородность**: $\|\lambda w\| = |\lambda| \cdot \|w\|$ для любого скаляра $\lambda$.
3.  **Неравенство треугольника**: $\|w + v\| \leq \|w\| + \|v\|$.

### Наиболее важные нормы в регуляризации:
1.  **$L_2$ норма (Евклидова)**: Измеряет расстояние по прямой. Используется в **Ridge**-регрессии.
    $$\|w\|_2 = \sqrt{\sum_{j=1}^d w_j^2}$$
2.  **$L_1$ норма (Манхэттенская)**: Сумма абсолютных значений. Используется в **Lasso**-регрессии.
    $$\|w\|_1 = \sum_{j=1}^d |w_j|$$

### Обыденный взгляд
Представьте, что вектор — это маршрут. **$L_2$ норма** — это расстояние, которое пролетит птица от старта до финиша по прямой. **$L_1$ норма** — это расстояние, которое проедет такси по сетке городских улиц (только повороты под 90 градусов). В регуляризации норма служит «штрафом»: мы заставляем модель минимизировать не только ошибку, но и «длину» вектора своих весов, чтобы она не становилась слишком сложной.


## Понятие регуляризации

`Регуляризация` — это совокупность техник, используемых в машинном обучении для предотвращения переобучения (оверфиттинга) модели и улучшения её обобщающей способности, то есть способности давать точные прогнозы на новых, невиданных ранее данных.

![alttext](../data/loss-vs-model-complexity.png)

In [2]:
runge_example()

NameError: name 'clear_output' is not defined

Как видно из примеров с полиномиальной регрессией, модель с большим количеством параметров (высокая степень полинома) может идеально подстроиться под обучающие данные, точно проходя через каждую точку. Однако такая модель начинает улавливать не только закономерности, но и шум в данных. В результате её поведение между точками обучения становится хаотичным, а коэффициенты модели принимают огромные по модулю значения (становятся «длинными»). Это и есть переобучение.

<div style='background-color: #ffeb3b; padding: 15px; border-left: 5px solid #ffc107; margin: 10px 0'>
<b>💡 Идея:</b> Давайте штрафовать модель за неверные предсказания
</div>


Таким образом, новая задача оптимизации выглядит так:

$$
\sum_{i=1}^\ell \mathcal{L}(w, x_i) + \lambda \mathcal{R}(w) \to \min_W \quad \text{(empirical risk minimization, ERM)}
$$

* $\mathcal{L}(w, x_i)$ - функция потерь (**лосс**)
* $\mathcal{R}(w)$ - регуляризатор
* $\lambda$ - гиперпараметр регуляризации, который балансирует между двумя целями:
    * Минимизация ошибки на обучающих данных (хорошее соответствие данным)
    * Минимизация сложности модели (предотвражение переобучения)

Чем больше $\lambda$, тем сильнее штраф за сложность и тем более простую модель мы получим. 

Подбирается $\lambda$ на валидационной выборке.

## Вероятностные истоки

Давайте вспомним **MAP**

**MAP:** максимизируем апостериорную вероятность:

$$\theta_{MAP} = \arg\max_{\theta} p(\theta | D) = \arg\max_{\theta} p(D | \theta)p(\theta)$$

`Какие параметры наиболее вероятны с учетом и данных, и того, что мы знали до эксперимента?`

Давайте посмотрим на регрессию с байесовской стороны. До сих пор в нашем анализе линейной регрессии участвовало только правдоподобие. Теперь введем априорное распределение, которое будет выражать наши представления о весах $w$. Выберем его в форме нормального распределения:
$$p(w) = \mathcal{N}(w \mid \mu_0, \Sigma_0).$$
Пусть у нас есть набор данных $X = (x_1, \dots, x_N)$ и ответы $y = (y_1, \dots, y_N)$. Предполагая, что данные i.i.d., правдоподобие запишется как:
$$p(y \mid X, w, \sigma^2) = \prod_{i=1}^N \mathcal{N}(y_i \mid w^\top x_i, \sigma^2) = \mathcal{N}(y \mid Xw, \sigma^2 \mathbb{I}).$$
Согласно теореме Байеса, апостериорное распределение пропорционально произведению правдоподобия на априорную плотность:
$$p(w \mid y, X) \propto p(y \mid X, w, \sigma^2) p(w).$$
Так как произведение нормальных плотностей дает нормальную плотность, мы можем записать $p(w \mid y) = \mathcal{N}(w \mid \mu_N, \Sigma_N)$. Чтобы найти его параметры, выпишем логарифм апостериорной плотности, отбрасывая константы, не зависящие от $w$:
$$\ln p(w \mid y, X) = -\frac{1}{2}(w - \mu_0)^\top \Sigma_0^{-1} (w - \mu_0) - \frac{1}{2\sigma^2} (y - Xw)^\top (y - Xw) + \text{const}.$$
Раскроем скобки, чтобы сгруппировать члены по степеням $w$:
$$\ln p(w \mid y, X) = -\frac{1}{2} \left[ w^\top \Sigma_0^{-1} w - 2w^\top \Sigma_0^{-1} \mu_0 + \frac{1}{\sigma^2} (y^\top y - 2w^\top X^\top y + w^\top X^\top X w) \right] + \text{const}.$$
Сгруппируем квадратичные члены ($w^\top [\dots] w$) и линейные члены ($w^\top [\dots]$):
$$\ln p(w \mid y, X) = -\frac{1}{2} \left[ w^\top \left( \Sigma_0^{-1} + \frac{1}{\sigma^2} X^\top X \right) w - 2w^\top \left( \Sigma_0^{-1} \mu_0 + \frac{1}{\sigma^2} X^\top y \right) \right] + \text{const}.$$
Сравнивая это с логарифмом плотности произвольного нормального распределения $\mathcal{N}(w \mid \mu_N, \Sigma_N)$, который имеет вид $-\frac{1}{2} (w^\top \Sigma_N^{-1} w - 2w^\top \Sigma_N^{-1} \mu_N)$, получаем искомые параметры:
$$\Sigma_N = \left( \Sigma_0^{-1} + \frac{1}{\sigma^2} X^\top X \right)^{-1}, \quad \mu_N = \Sigma_N \left( \Sigma_0^{-1} \mu_0 + \frac{1}{\sigma^2} X^\top y \right).$$
Если теперь взять априорное распределение с центром в нуле $p(w) = \mathcal{N}(w \mid \mathbf{0}, \frac{1}{\alpha} \mathbb{I})$, то $\mu_0 = 0$ и $\Sigma_0^{-1} = \alpha \mathbb{I}$. В этом случае логарифм апостериорного распределения примет вид:
$$\ln p(w \mid y) = -\frac{1}{2\sigma^2} \sum_{i=1}^N \left( y_i - w^\top x_i \right)^2 - \frac{\alpha}{2} w^\top w + \text{const}.$$



## Регуляризация линейных моделей

### L2-регуляризация (Ridge \ Гребневая регрессия \ Регуляризация по Тихонову)

В случае, если наши веса модели распределены нормально, из прошлой формулы для задачи регрессии, мы получим:

$$\mathcal{L} = \frac{1}{N}\sum_{i=1}^n(y_i - w^\top x_i)^2 + \lambda \|w\|_2^2 = MSE + + \lambda \|w\|_2^2$$

Как мы видели выше из байесовского подхода, это эквивалентно поиску моды апостериорного распределения (MAP) при условии, что мы заранее верим: веса распределены нормально вокруг нуля. Чем больше $\lambda$, тем сильнее мы «стягиваем» веса к нулю, не давая им принимать огромные значения.

С `математической точки` зрения это ограничение задает область в форме L2-шара, который в двумерном пространстве выглядит как идеальный круг с центром в начале координат: $w_1^2 + w_2^2 \leq R^2$. 

Из-за отсутствия «острых углов» на границе, линии уровня функции ошибки (эллипсы), раздуваясь из центра, касаются штрафной области в произвольной точке окружности. Шанс того, что точка касания попадет ровно на ось координат, практически равен нулю, поэтому веса в Ridge-регрессии становятся очень маленькими, но почти никогда не становятся **равными нулю**, сохраняя вклад каждого признака в итоговый прогноз. 

С `обыденной точки` зрения это похоже на мягкий менеджмент в кризис: вместо того чтобы увольнять целые отделы, вы пропорционально сокращаете бюджеты всем сотрудникам. Никто не уходит с работы, но все начинают действовать более экономно и осторожно, что делает всю систему более устойчивой к внешним шокам и шумам в данных.


In [ ]:
interact(plot_ridge, alpha=FloatLogSlider(value=1e-9, base=10, min=-9, max=2, step=0.5, description='Lambda'));

interactive(children=(FloatLogSlider(value=1e-09, description='Lambda', max=2.0, min=-9.0, step=0.5), Output()…

: 

![alt](../data/q.webp)

### L1-регуляризация (Least Absolute Shrinkage and Selection Operator - LASSO)

В **L1-регуляризации (Lasso)** штраф добавляется не за квадрат весов, а за их абсолютные значения: $$\mathcal{L} = MSE + \lambda \|w\|_1.$$ С математической точки зрения это ограничение задает область в форме L1-шара, который в двумерном пространстве выглядит как ромб с вершинами, лежащими строго на осях координат: $|w_1| + |w_2| \leq R$. 

Из-за наличия «острых углов» на осях, линии уровня функции ошибки (эллипсы), раздуваясь из центра, с очень высокой вероятностью касаются штрафной области именно в одной из этих вершин. В такой точке касания одна или несколько координат весов становятся **равными нулю**, что делает Lasso мощным инструментом для автоматического отбора признаков (feature selection). 

С `обыденной точки` зрения это похоже на жесткий менеджмент: если у вас ограниченный бюджет, вы не просто «всем урезаете зарплату на 5%» (как в Ridge), а полностью закрываете неэффективные отделы, оставляя только те, что приносят реальную прибыль.


In [ ]:
interact(plot_lasso, alpha=FloatLogSlider(value=1e-4, base=10, min=-5, max=0, step=0.5, description='Lambda'));

![alttext](../data/reg.webp)

### ElasticNet (L1 + L2)

**ElasticNet** представляет собой гибрид, который объединяет сильные стороны обоих подходов через линейную комбинацию штрафов: $$\mathcal{L}_{EN}(w) = MSE + \lambda_1 \left( \lambda_2 \|w\|_1 + \frac{1-\lambda_2}{2} \left \lVert w \right \rVert_2^2 \right),$$ где параметр $\lambda_2$  регулирует баланс между «ромбом» и «кругом». 

`Математически` это создает форму «выпуклого ромба» со скругленными гранями. Это критически важно, когда в данных есть сильно связанные (коррелированные) признаки. В то время как Lasso может случайно выбрать один признак из группы и занулить остальные, ElasticNet стремится сохранить всю группу, распределив веса между ними, но при этом всё еще эффективно избавляется от абсолютно бесполезных переменных. 

В жизни это стратегия «здравого смысла»: вы избавляетесь от явного мусора, но не выбрасываете запасное колесо только потому, что сегодня оно не крутится, но когда-то может и пригодится..


### Вывод в одну строку:

* **L1 (Lasso)** — зануляет веса (делает разреженные модели); 
* **L2 (Ridge)** — уменьшает веса (делает устойчивые модели);
* **ElasticNet** — зануляет мусор, но сохраняет группы связанных признаков.


# Источники